In [35]:
import pandas as pd
import numpy as np
import sys
import os

# 1. Get the current directory of the notebook (src/Trainers)
current_dir = os.getcwd()

# 2. Go up two levels to get the Project Root (SDHAR-Dataset-Project)
#    Up once = src
#    Up twice = SDHAR-Dataset-Project
project_root = os.path.abspath(os.path.join(current_dir, '..', '..'))

# 3. Add this path to Python so it can see the 'src' package
if project_root not in sys.path:
    sys.path.append(project_root)

print(f"Project Root added: {project_root}")

from tensorflow.keras.models import load_model

input_path = "../../processed_data/SDHAR/final_processed_data_ALL_DAYS.csv"

df = pd.read_csv("../../processed_data/SDHAR/final_processed_data_ALL_DAYS.csv")

def create_windows(X, y, window_size, step_size):
    X_win, y_win = [], []
    for i in range(0, len(X) - window_size, step_size):
        window = X.iloc[i:i + window_size].values
        label = y.iloc[i + window_size]
        X_win.append(window)
        y_win.append(label)
    return np.array(X_win), np.array(y_win)

# The activity column your models were trained on
LABEL_COL = "activity_user_1"  # <-- change if needed

Project Root added: C:\Users\Popey\CSCI Data Mining Project\SDHAR-Dataset-Project


In [10]:
#Processed Data splitter to help with accessing it manually.
n_rows = len(df)
mid = n_rows // 2  # integer division

print(f"Total rows: {n_rows}, first part: {mid}, second part: {n_rows - mid}")

# Build output paths in the same directory
base_dir = os.path.dirname(os.path.abspath(input_path))
part1_path = os.path.join(base_dir, "final_processed_data_ALL_DAYS_part1.csv")
part2_path = os.path.join(base_dir, "final_processed_data_ALL_DAYS_part2.csv")

# Split and save
df.iloc[:mid].to_csv(part1_path, index=False)
df.iloc[mid:].to_csv(part2_path, index=False)

print("Done! Saved:")
print("  ", part1_path)
print("  ", part2_path)

Total rows: 2664474, first part: 1332237, second part: 1332237
Done! Saved:
   C:\Users\Popey\CSCI Data Mining Project\SDHAR-Dataset-Project\processed_data\SDHAR\final_processed_data_ALL_DAYS_part1.csv
   C:\Users\Popey\CSCI Data Mining Project\SDHAR-Dataset-Project\processed_data\SDHAR\final_processed_data_ALL_DAYS_part2.csv


In [36]:
def build_activity_segments(df, label_col=LABEL_COL):
    """
    Find contiguous segments where label_col stays the same (ignoring NaNs).
    Returns dict: label_value -> list of (start_idx, end_idx) (end exclusive).
    """
    labels = df[label_col].values
    segments_by_label = {}
    start_idx = None
    current_label = None

    for idx, label in enumerate(labels):
        if pd.isna(label):
            if current_label is not None:
                segments_by_label.setdefault(current_label, []).append((start_idx, idx))
                current_label = None
                start_idx = None
            continue

        label_int = int(label)

        if current_label is None:
            current_label = label_int
            start_idx = idx
        elif label_int != current_label:
            segments_by_label.setdefault(current_label, []).append((start_idx, idx))
            current_label = label_int
            start_idx = idx

    if current_label is not None and start_idx is not None:
        segments_by_label.setdefault(current_label, []).append((start_idx, len(labels)))

    return segments_by_label

segments_by_label = build_activity_segments(df, LABEL_COL)

print("Activity IDs found:", sorted(int(x) for x in segments_by_label.keys()))


def script_names_to_numeric(script_by_name, name_to_id, seconds_per_row=2.0):
    """
    Convert a user script written in terms of activity names to numeric labels and row lengths.

    script_by_name: list of dicts, each like:
        {"activity": "Sleep", "minutes": 60}
        or {"activity": "Prepare_Breakfast", "rows": 1200}
    """
    numeric_script = []
    for step in script_by_name:
        name = step["activity"]
        if name not in name_to_id:
            raise ValueError("Unknown activity name %r. Known names: %s"
                             % (name, list(name_to_id.keys())))
        label_id = name_to_id[name]

        # Allow user to specify length in minutes OR rows
        if "rows" in step:
            length_rows = int(step["rows"])
        elif "minutes" in step:
            # convert minutes -> rows using seconds_per_row
            length_rows = int((step["minutes"] * 60.0) / seconds_per_row)
        else:
            raise ValueError("Each step must have 'rows' or 'minutes'.")

        numeric_script.append({"label": label_id, "length_rows": length_rows})

    return numeric_script

def build_synthetic_from_numeric_script(
    df,
    segments_by_label,
    script,
    label_col=LABEL_COL,
    random_state=None,
    recompute_time_features=True,
    seconds_per_row=2.0,
):
    """
    script: list of dicts: {"label": <int_id>, "length_rows": <int>}
    """
    rng = np.random.default_rng(random_state)
    pieces = []

    for step in script:
        label = int(step["label"])
        target_len = int(step["length_rows"])

        if label not in segments_by_label:
            raise ValueError("No segments found for label %r" % label)

        segs = segments_by_label[label]
        length_left = target_len

        while length_left > 0:
            start, end = segs[rng.integers(len(segs))]
            seg_len = end - start
            if seg_len <= 0:
                continue

            take_len = min(seg_len, length_left)
            piece = df.iloc[start:start + take_len]
            pieces.append(piece)
            length_left -= take_len

    synthetic = pd.concat(pieces, ignore_index=True)

    # Optionally recompute sin_time / cos_time to be smooth over this new sequence
    if recompute_time_features and "sin_time" in synthetic.columns and "cos_time" in synthetic.columns:
        n = len(synthetic)
        t = np.arange(n) * seconds_per_row
        angles = 2 * np.pi * (t % 86400) / 86400.0  # one full cycle per (24h)
        synthetic["sin_time"] = np.sin(angles)
        synthetic["cos_time"] = np.cos(angles)

    return synthetic

def build_synthetic_from_name_script(
    df,
    segments_by_label,
    script_by_name,
    activity_name_to_id,
    label_col=LABEL_COL,
    random_state=None,
    recompute_time_features=True,
    seconds_per_row=2.0,
):
    numeric_script = script_names_to_numeric(
        script_by_name,
        activity_name_to_id,
        seconds_per_row=seconds_per_row,
    )
    return build_synthetic_from_numeric_script(
        df=df,
        segments_by_label=segments_by_label,
        script=numeric_script,
        label_col=label_col,
        random_state=random_state,
        recompute_time_features=recompute_time_features,
        seconds_per_row=seconds_per_row,
    )


Activity IDs found: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]


In [39]:
activity_names = [
    "BATHROOM ACTIVITY", "CHORES", "COOK", "DISHWASHING", "DRESS", "EAT", "LAUNDRY",
    "MAKE SIMPLE FOOD", "OUT HOME", "PET", "READ", "RELAX", "SHOWER", "SLEEP",
    "TAKE MEDS", "WATCH TV", "WORK", "OTHER"
]

activity_id_to_name = {i: name for i, name in enumerate(activity_names)}
activity_name_to_id = {name: i for i, name in enumerate(activity_names)}

# Example: 8 hours of Sleep, 30 minutes of Prepare_Breakfast, 20 minutes of Eat_Breakfast
activity_script_by_name = [
    {"activity": "DISHWASHING",             "minutes": 30},
    {"activity": "BATHROOM ACTIVITY", "minutes": 30},
    {"activity": "COOK",     "minutes": 20},
    {"activity": "SHOWER",     "minutes": 20}
]

synthetic_df = build_synthetic_from_name_script(
    df=df,
    segments_by_label=segments_by_label,
    script_by_name=activity_script_by_name,
    activity_name_to_id=activity_name_to_id,
    random_state=42,
    recompute_time_features=False,
    seconds_per_row=2.0,  # ~2 seconds per row in your data
)

print(synthetic_df.shape)
print(synthetic_df[LABEL_COL].value_counts().sort_index())
synthetic_df.head()


(3000, 43)
activity_user_1
0.0     900
2.0     600
3.0     900
12.0    600
Name: count, dtype: int64


,activity_user_1,activity_user_2,c1,c2,c3,c4,c5,c6,c7,c8,...,v2,v3,v4,v5,v6,v7,v8,v9,sin_time,cos_time
0,3.0,8.0,0,0.0,0,0,0,0,0.0,0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.123168,-0.992386
1,3.0,8.0,1,0.0,0,0,0,0,0.0,0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.123024,-0.992404
2,3.0,8.0,1,0.0,0,0,0,0,0.0,0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.122880,-0.992422
3,3.0,8.0,1,0.0,0,0,0,0,0.0,0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.122735,-0.992439
4,3.0,8.0,1,0.0,0,0,0,0,0.0,0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.122591,-0.992457


In [40]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import MinMaxScaler  # change to StandardScaler if you used that
from tensorflow.keras.models import load_model
import joblib

In [43]:
import joblib
from tensorflow.keras.models import load_model
from sklearn.preprocessing import MinMaxScaler  # or StandardScaler, depending on what you used

# If you DON'T already have create_windows defined, uncomment this:
# def create_windows(X, y, window_size, step_size):
#     X_win, y_win = [], []
#     for i in range(0, len(X) - window_size, step_size):
#         window = X.iloc[i:i + window_size].values
#         label = y.iloc[i + window_size]
#         X_win.append(window)
#         y_win.append(label)
#     return np.array(X_win), np.array(y_win)

# ----- 1. Config -----
WINDOW_SIZE = 60   # <-- change to what you used in training
STEP_SIZE   = 30   # <-- change to what you used in training

# Paths to your saved models (EDIT THESE)
dt_first_path   = "../../models/SDHAR/DecisionTree_first_iteration.joblib"
dt_norm_path    = "../../models/SDHAR/DecisionTree_normzlized.joblib"
rf_first_path   = "../../models/SDHAR/RandomForest_first_iteration.joblib"
rf_norm_path    = "../../models/SDHAR/RandomForest_normalized.joblib"
lstm_first_path = "../../models/SDHAR/LSTM_first_iteration.keras"
lstm_norm_path  = "../../models/SDHAR/LSTM_normalized.keras"

# ----- 2. Prepare features + labels from synthetic_df -----
df = synthetic_df.copy()  # rename if your variable is different

label_col = "activity_user_1"
feature_cols = [c for c in df.columns if "activity" not in c.lower()]

# Use only rows where we actually have a label
mask = df[label_col].notna()
X_raw = df.loc[mask, feature_cols].reset_index(drop=True)
y_raw = df.loc[mask, label_col].astype(int).reset_index(drop=True)

# ----- 3. Build windows for "first iteration" models (no normalization) -----
X_win, y_win = create_windows(X_raw, y_raw, WINDOW_SIZE, STEP_SIZE)

# Flatten for tree models, keep 3D for LSTM
n_samples, n_steps, n_feats = X_win.shape
X_flat = X_win.reshape(n_samples, n_steps * n_feats)

# ----- 4. Optionally build normalized version for *_normalized models -----
# Here we just fit a scaler on X_raw to put features on 0-1 range.
# If you used StandardScaler in training, replace MinMaxScaler with StandardScaler.
scaler = MinMaxScaler()
scaler.fit(X_raw)

X_scaled = pd.DataFrame(scaler.transform(X_raw), columns=feature_cols)
X_win_scaled, _ = create_windows(X_scaled, y_raw, WINDOW_SIZE, STEP_SIZE)

n_samples2, n_steps2, n_feats2 = X_win_scaled.shape
X_flat_scaled = X_win_scaled.reshape(n_samples2, n_steps2 * n_feats2)

# ----- 5. Load models -----
dt_first   = joblib.load(dt_first_path)
dt_norm    = joblib.load(dt_norm_path)
rf_first   = joblib.load(rf_first_path)
rf_norm    = joblib.load(rf_norm_path)
lstm_first = load_model(lstm_first_path)
lstm_norm  = load_model(lstm_norm_path)

# ----- 6. Predict with all models -----
# Tree models on flattened windows
y_dt_first   = dt_first.predict(X_flat)
y_dt_norm    = dt_norm.predict(X_flat_scaled)
y_rf_first   = rf_first.predict(X_flat)
y_rf_norm    = rf_norm.predict(X_flat_scaled)

# LSTMs on 3D windowsx
y_lstm_first = np.argmax(lstm_first.predict(X_win), axis=1)
y_lstm_norm  = np.argmax(lstm_norm.predict(X_win_scaled), axis=1)

# ----- 7. Build a simple results DataFrame (per window) -----
# Figure out which row each window's label corresponds to (end of window)
window_end_indices = list(range(WINDOW_SIZE, WINDOW_SIZE + STEP_SIZE * len(y_win), STEP_SIZE))

if "timestamp" in df.columns:
    times = df.loc[mask, "timestamp"].iloc[window_end_indices].reset_index(drop=True)
else:
    times = pd.Series(window_end_indices, name="row_index")

results = pd.DataFrame({
    "time": times,
    "true_activity": y_win.astype(int),
    "dt_first": y_dt_first.astype(int),
    "dt_norm": y_dt_norm.astype(int),
    "rf_first": y_rf_first.astype(int),
    "rf_norm": y_rf_norm.astype(int),
    "lstm_first": y_lstm_first.astype(int),
    "lstm_norm": y_lstm_norm.astype(int),
})

results

4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 412ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 3s 593ms/step


,time,true_activity,dt_first,dt_norm,rf_first,rf_norm,lstm_first,lstm_norm
0,60,3,3,5,3,7,3,5
1,90,3,3,5,3,7,8,5
2,120,3,3,5,3,17,17,5
3,150,3,3,5,3,17,17,5
4,180,3,3,5,3,17,2,5
...,...,...,...,...,...,...,...,...
93,2850,12,12,12,12,12,12,12
94,2880,12,12,7,12,12,12,12
95,2910,12,12,7,12,17,12,12
96,2940,12,12,12,12,17,12,12


In [44]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import numpy as np

# Activity mapping (same order you used when training)
activity_names = [
    "BATHROOM ACTIVITY", "CHORES", "COOK", "DISHWASHING", "DRESS", "EAT",
    "LAUNDRY", "MAKE SIMPLE FOOD", "OUT HOME", "PET", "READ", "RELAX",
    "SHOWER", "SLEEP", "TAKE MEDS", "WATCH TV", "WORK", "OTHER"
]

# True labels for each window (from your previous cell)
y_true = y_win.astype(int)

# Collect all model predictions that you computed in the previous cell
model_preds = {
    "DecisionTree_first_iteration": y_dt_first,
    "DecisionTree_normalized":      y_dt_norm,
    "RandomForest_first_iteration": y_rf_first,
    "RandomForest_normalized":      y_rf_norm,
    "LSTM_first_iteration":         y_lstm_first,
    "LSTM_normalized":              y_lstm_norm,
}

# For classification_report we’ll fix the label set 0..17 so rows line up
all_labels = list(range(len(activity_names)))

accuracies = {}

for name, y_pred in model_preds.items():
    y_pred = np.asarray(y_pred).astype(int)
    acc = accuracy_score(y_true, y_pred)
    accuracies[name] = acc

    print("\n" + "=" * 70)
    print(f"{name}")
    print(f"Accuracy on synthetic data: {acc:.4f}\n")

    print("Classification report:")
    print(classification_report(
        y_true,
        y_pred,
        labels=all_labels,
        target_names=activity_names,
        zero_division=0,   # avoid warnings for labels not present
    ))

    print("Confusion matrix (rows = true, cols = predicted):")
    print(confusion_matrix(y_true, y_pred, labels=all_labels))

# Which model is best by accuracy?
best_model = max(accuracies, key=accuracies.get)
print("\n" + "#" * 70)
print("Best model on this synthetic dataset (by accuracy):")
print(f"{best_model} with accuracy {accuracies[best_model]:.4f}")
print("#" * 70)



DecisionTree_first_iteration
Accuracy on synthetic data: 0.7551

Classification report:
                   precision    recall  f1-score   support

BATHROOM ACTIVITY       0.69      0.90      0.78        30
           CHORES       0.00      0.00      0.00         0
             COOK       0.95      0.90      0.92        20
      DISHWASHING       1.00      0.82      0.90        28
            DRESS       0.00      0.00      0.00         0
              EAT       0.00      0.00      0.00         0
          LAUNDRY       0.00      0.00      0.00         0
 MAKE SIMPLE FOOD       0.00      0.00      0.00         0
         OUT HOME       0.00      0.00      0.00         0
              PET       0.00      0.00      0.00         0
             READ       0.00      0.00      0.00         0
            RELAX       0.00      0.00      0.00         0
           SHOWER       1.00      0.30      0.46        20
            SLEEP       0.00      0.00      0.00         0
        TAKE MEDS       0